<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Ventaja arancelaria de Colombia frente a otros proveedores

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.

In [1]:
import pandas as pd
import numpy as np

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import temp, tables

### Lectura de datos

Carga la base de aranceles bilaterales, la cartera con tamaño de mercado y los aranceles
anunciados por Estados Unidos.

In [3]:
df_aranceles = pd.read_parquet(temp / 'df_aranceles.parquet')
df_trump = pd.read_parquet(temp / 'df_trump.parquet')

df_products = pd.read_parquet(temp / 'df_products_mercado.parquet')
df_products['HS4'] = df_products['HS6'].str[:4]
df_products['HS2'] = df_products['HS6'].str[:2]

carpeta_tariffs = tables / 'tariffs'

### Cálculo de la ventaja arancelaria

Compara el arancel que cada país reportante aplica a Colombia con el promedio ponderado que
aplica al resto de proveedores del mismo producto. La ventaja es uno cuando el arancel que
enfrenta Colombia es menor o igual al de sus competidores.

La imputación completa el valor de los productos sin dato a seis dígitos con el promedio de su
partida de cuatro y, en su defecto, con el de su capítulo de dos.

El notebook original repetía este bloque completo dos veces, una por escenario; aquí se define
una vez como par de funciones.

In [4]:
def ventaja_arancelaria(df_base):
    """Calcula la ventaja arancelaria de Colombia frente al resto de proveedores.

    Compara el arancel que cada país reportante aplica a Colombia con el promedio
    ponderado que aplica a los demás proveedores del mismo producto.
    """
    df = df_base.copy()

    df['total_imports'] = df.groupby(['ISO Reporter','Product'])['Imports Value in 1000 USD'].transform('sum')
    df['total_imports_sin_col'] = df['total_imports'] - df['Imports Value in 1000 USD']
    df_col = df.query("`ISO Partner` == 'COL'").reset_index(drop=True)

    list_col = df_base.query("`ISO Partner` == 'COL'")['Product'].unique().tolist()
    df_sin_col = df_base.query("`ISO Partner` != 'COL' and Product in @list_col")

    df_sin_col = df_sin_col.merge(df_col[['ISO Reporter','Product', 'total_imports_sin_col']], on=['ISO Reporter','Product'], how='left').dropna(subset=['total_imports_sin_col']).sort_values(by=['Product','ISO Reporter']).reset_index(drop=True)
    df_sin_col['ponderacion'] = df_sin_col['Imports Value in 1000 USD'] / df_sin_col['total_imports_sin_col']
    df_sin_col['tariff ponderado'] = df_sin_col['Weighted Average'] * df_sin_col['ponderacion']
    df_sin_col = df_sin_col.groupby(['Reporter Name', 'Product'])['tariff ponderado'].sum().reset_index().rename(columns={'tariff ponderado': 'promedio ponderado'})

    df_col = df_col.merge(df_sin_col, on=['Reporter Name', 'Product'], how='left').drop(columns=['total_imports','total_imports_sin_col','Imports Value in 1000 USD']).fillna({'promedio ponderado': 0})
    df_col['tariff'] = (1 + df_col['promedio ponderado']) / (1 + df_col['Weighted Average'])
    df_col['Ventaja Arancelaria'] = np.where(df_col['tariff'] >= 1, 1, 0)

    df_col['HS6'] = df_col['Product']
    df_col['HS4'] = df_col['HS6'].str[:4]
    df_col['HS2'] = df_col['HS6'].str[:2]
    return df_col


def imputar_ventaja(df_productos, df_col):
    """Asigna la ventaja arancelaria por HS6 y la completa con el promedio de HS4 y de HS2."""
    df = df_productos.merge(
        df_col.groupby(['HS6'])['Ventaja Arancelaria'].mean().reset_index().round(3), how='left', on='HS6'
    ).merge(
        df_col.groupby(['HS4'])['Ventaja Arancelaria'].mean().reset_index().round(3), how='left', on='HS4'
    ).merge(
        df_col.groupby(['HS2'])['Ventaja Arancelaria'].mean().reset_index().round(3).rename(columns={'Ventaja Arancelaria': 'Ventaja Arancelaria_z'}), how='left', on='HS2'
    )
    df['Ventaja Arancelaria'] = df['Ventaja Arancelaria_x'].fillna(df['Ventaja Arancelaria_y']).fillna(df['Ventaja Arancelaria_z'])
    return df.drop(columns=['Ventaja Arancelaria_x', 'Ventaja Arancelaria_y', 'Ventaja Arancelaria_z'])

### Escenarios

Genera los dos escenarios: el de los aranceles aplicados vigentes y el que incorpora los
aranceles anunciados por Estados Unidos, restringido a ese país reportante.

In [5]:
# escenario sin los aranceles anunciados
df_aranceles_col = ventaja_arancelaria(df_aranceles)
df_salida = imputar_ventaja(df_products, df_aranceles_col)

carpeta_tariffs.mkdir(parents=True, exist_ok=True)
df_salida.drop(columns=['Fuente','Categoría']).drop_duplicates().sort_values(['Margen','Potencial Exportador Restante','Ventaja Arancelaria'], ascending=[True, False, False]).reset_index(drop=True).to_excel(carpeta_tariffs / 'df_tariffs.xlsx', index=False)

# escenario con los aranceles anunciados por Estados Unidos, sumados al arancel aplicado
df_aranceles_con_trump = df_aranceles.copy()
df_aranceles_con_trump = df_aranceles_con_trump.merge(df_trump, how='left', left_on=['ISO Reporter','ISO Partner'], right_on=['ISO Reporter','ISO Partner'])
df_aranceles_con_trump['Weighted Average'] = df_aranceles_con_trump['Arancel'].fillna(0) + df_aranceles_con_trump['Weighted Average']

df_aranceles_col_con_trump = ventaja_arancelaria(df_aranceles_con_trump).query("`ISO Reporter` == 'USA'")
df_salida_trump = imputar_ventaja(df_products, df_aranceles_col_con_trump).fillna({'Ventaja Arancelaria': 0})

df_salida_trump.drop(columns=['Fuente','Categoría']).drop_duplicates().sort_values(['Margen','Potencial Exportador Restante','Ventaja Arancelaria'], ascending=[True, False, False]).reset_index(drop=True).to_excel(carpeta_tariffs / 'df_tariffs_trump.xlsx', index=False)